In [ ]:
!pip install xgboost -q

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Train.csv to Train.csv


Needed requirements:

SKLEARN: test,training,preprocessing

xgboost: classifier with xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Train.csv')
df.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


The preprocessing cell cleans the data by filling missing values with the median/mode, clipping outliers using the IQR method, and encoding categorical variables (gender as binary, product importance as ordinal, warehouse/shipment mode as one-hot). It then balances the dataset by randomly removing On Time records to match the Not on Time count, splits into train/test sets, and standardizes the continuous features using StandardScaler fit only on the training data to prevent data leakage.

In [ ]:
df.drop(columns=['ID'], inplace=True)

for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in ['Customer_care_calls', 'Cost_of_the_Product', 'Discount_offered', 'Weight_in_gms']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    df[col] = df[col].clip(Q1-1.5*(Q3-Q1), Q3+1.5*(Q3-Q1))

df['Gender'] = (df['Gender'] == 'M').astype(int)
df['Product_importance'] = df['Product_importance'].map({'low': 0, 'medium': 1, 'high': 2})
cat_dummies = pd.get_dummies(df[['Warehouse_block', 'Mode_of_Shipment']], drop_first=True).astype(float)

target = 'Reached.on.Time_Y.N'
numeric_features = [
    'Customer_care_calls', 'Cost_of_the_Product', 'Prior_purchases',
    'Product_importance', 'Customer_rating', 'Gender',
    'Discount_offered', 'Weight_in_gms',
]

X_full = pd.concat([df[numeric_features], cat_dummies], axis=1)
y_full = df[target]

df_bal = X_full.copy()
df_bal[target] = y_full.values
minority = df_bal[df_bal[target] == 0]
majority = df_bal[df_bal[target] == 1].sample(n=len(minority), random_state=42)
balanced = pd.concat([minority, majority]).sample(frac=1, random_state=42)
X = balanced.drop(columns=[target])
y = balanced[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scale_cols = ['Customer_care_calls', 'Cost_of_the_Product', 'Prior_purchases',
              'Discount_offered', 'Weight_in_gms']
scaler = StandardScaler()
X_train = X_train.copy(); X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

Output of preprocessed cell

In [ ]:
print(f'Shape: {X.shape}')
print(f'Class balance — 0: {(y==0).sum()}  1: {(y==1).sum()}')
print()
X.head()

Shape: (8872, 14)
Class balance — 0: 4436  1: 4436



,Customer_care_calls,Cost_of_the_Product,Prior_purchases,Product_importance,Customer_rating,Gender,Discount_offered,Weight_in_gms,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Road,Mode_of_Shipment_Ship
3616,3,142,3,0,3,0,7,4439,0.0,1.0,0.0,0.0,0.0,1.0
5059,4,143,3,0,3,0,2,4693,0.0,0.0,0.0,1.0,1.0,0.0
9711,4,217,5,0,5,0,3,1572,1.0,0.0,0.0,0.0,1.0,0.0
8577,5,230,3,1,4,1,6,4257,1.0,0.0,0.0,0.0,0.0,1.0
3529,3,197,3,0,5,1,3,5658,0.0,0.0,0.0,1.0,0.0,1.0


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)
probs = xgb.predict_proba(X_test)[:, 1]

The threshold is the probability cutoff for predicting "On Time".

XGBoost doesn't directly output 0 or 1 — it outputs a probability between 0 and 1 (e.g. 0.63 = 63% chance of On Time). The threshold decides where you draw the line:

Finding the best threshold, depending on the best parameters.

In [ ]:
thresholds = np.arange(0.30, 0.71, 0.05)
rows = []
for t in thresholds:
    p = (probs >= t).astype(int)
    cm = confusion_matrix(y_test, p)
    recall_0 = cm[0,0] / (cm[0,0] + cm[0,1] + 1e-9)
    recall_1 = cm[1,1] / (cm[1,1] + cm[1,0] + 1e-9)
    rows.append({
        'Threshold':        round(t, 2),
        'Accuracy':         round((p == y_test.values).mean(), 4),
        'Recall_NotOnTime': round(recall_0, 4),
        'Recall_OnTime':    round(recall_1, 4),
        'Recall_Gap':       round(abs(recall_0 - recall_1), 4),
        'F1_macro':         round(f1_score(y_test, p, average='macro'), 4),
    })

thresh_df = pd.DataFrame(rows)
thresh_df

,Threshold,Accuracy,Recall_NotOnTime,Recall_OnTime,Recall_Gap,F1_macro
0,0.30,0.5989,0.4054,0.7926,0.3872,0.5833
1,0.35,0.6366,0.5912,0.6821,0.0909,0.6359
2,0.40,0.6800,0.7466,0.6133,0.1333,0.6786
3,0.45,0.6935,0.8390,0.5479,0.2910,0.6869
4,0.50,0.7127,0.9122,0.5130,0.3992,0.7007
5,0.55,0.7239,0.9538,0.4938,0.4600,0.7085
6,0.60,0.7262,0.9764,0.4758,0.5006,0.7078
7,0.65,0.7296,0.9876,0.4713,0.5164,0.7102
8,0.70,0.7313,0.9955,0.4667,0.5288,0.7110


In [ ]:
best_t = thresh_df.loc[thresh_df['Recall_Gap'].idxmin(), 'Threshold']
print(f'Best threshold (balanced recall): {best_t}')

preds = (probs >= best_t).astype(int)
print(f'Accuracy: {(preds == y_test.values).mean():.4f}')
print()
print(classification_report(y_test, preds, target_names=['Not on Time', 'On Time']))

Best threshold (balanced recall): 0.35
Accuracy: 0.6366

              precision    recall  f1-score   support

 Not on Time       0.65      0.59      0.62       888
     On Time       0.62      0.68      0.65       887

    accuracy                           0.64      1775
   macro avg       0.64      0.64      0.64      1775
weighted avg       0.64      0.64      0.64      1775



In [ ]:
cm = confusion_matrix(y_test, preds)
pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1'])

,Pred 0,Pred 1
Actual 0,525,363
Actual 1,282,605


In [ ]:
fi = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
fi.round(4)

,0
Discount_offered,0.3306
Prior_purchases,0.0831
Weight_in_gms,0.0814
Cost_of_the_Product,0.0546
Warehouse_block_D,0.0501
Warehouse_block_C,0.0467
Warehouse_block_B,0.0461
Customer_care_calls,0.0460
Mode_of_Shipment_Road,0.0452
Customer_rating,0.0447


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  eval_metric='logloss', random_state=42, n_jobs=-1),
    X, y, cv=skf, scoring='accuracy'
)
for i, s in enumerate(cv_scores, 1):
    print(f'Fold {i}: {s:.4f}')
print(f'\nMean : {cv_scores.mean():.4f}')
print(f'Std  : {cv_scores.std():.4f}')

Fold 1: 0.7014
Fold 2: 0.7268
Fold 3: 0.7182
Fold 4: 0.7193
Fold 5: 0.7114

Mean : 0.7154
Std  : 0.0085
